
# The **edunn** library

[edunn](https://github.com/facundoq/edunn) is a library for defining and training neural networks built on [Numpy](https://numpy.org/), designed to be simple to _understand_. 

Even more importantly, it was designed to be simple to _implement_. In other words, its **main** purpose is to serve as a learning object for understanding how modern neural networks are implemented in frameworks such as [Keras](https://keras.io/) or [Pytorch](https://pytorch.org/). 

That said, it is also simple to _use_. For example, to define and train a three-layer neural network for classification with different activation functions, we can write code that looks very much like the code for those frameworks:

```python
import edunn as nn

dataset_name="iris"
x,y,classes = nn.datasets.load_classification(dataset_name)
n,din=x.shape
n_classes=y.max()+1


# Model definition
layers = [nn.Linear(din,10),
          nn.Bias(10),
          nn.ReLU(),

          nn.Linear(10,n_classes),
          nn.Bias(n_classes),
          nn.Softmax()
          ]

model = nn.Sequential(layers)
print("Network architecture:")
print(model.summary())

error = nn.MeanError(nn.CrossEntropyWithLabels())
# Optimization algorithm
optimizer = nn.GradientDescent(lr=0.1,epochs=3000,batch_size=32)

# Optimization algorithm
print("Training the network with gradient descent:")
history = optimizer.optimize(model,x,y,error)

# Performance report
y_pred = model.forward(x)
y_pred_labels = y_pred.argmax(axis=1)
print(f"Final model accuracy on the training set: {nn.metrics.accuracy(y,y_pred_labels)*100:0.2f}%")

```

# Prerequisites
To be able to implement the library, we assume you have already acquired the basic concepts of neural networks: 

* Layers
    * Linear layers
    * Activation functions
    * Composition of layers
    * The forward and backward methods
* The backpropagation algorithm
* Gradient descent
    * Computing gradients
    * Basic gradient-based optimization
* Batch computation/training

We also assume knowledge of Python and Numpy, as well as of working with tabular and image datasets.


# Components of the library

We describe the basic components of the library used in the code above, in order to provide context for the exercises ahead. 




# The **datasets** module


The `edunn.datasets` module lets you easily load a few sample datasets. These datasets will be used to check and experiment with the models.


In [ ]:
import edunn as nn
dataset_name="study2d"
x,y,classes = nn.datasets.load_classification(dataset_name)
x -= x.mean(axis=0)
x /= x.std(axis=0)
n,din=x.shape
n_classes=y.max()+1

print(f"The {dataset_name} dataset has {n} examples, {din} features per example and {n_classes} classes: {classes}.")


To see which other classification or regression datasets the `datasets` module of `edunn` provides (which we access as `nn.datasets`), you can run `nn.datasets.get_classification_names()` and `nn.datasets.get_regression_names()` and get a list of names.

In [ ]:
print("The available classification datasets are:")
print(nn.datasets.get_classification_names())
print()

print("The available regression datasets are:")
print(nn.datasets.get_regression_names())
print()

# edunn classes and modules

To use `edunn`, we import the library and name it `nn` so that it is easier to type.
```python
import edunn as nn
```

The library has one fundamental class, `Model`. It is the superclass of the models/layers we are going to implement, and it defines two abstract methods for its subclasses to implement:

* `forward(x)`: computes the output `y` given an input `x`. 
    * As a simplifying assumption for the library, models can only have one input and one output, and both must be numpy arrays (except for the error models). In practice, we will see that this is not a significant limitation.
* `backward(δEδy)`: computes the gradient of the error with respect to the input (`δEδx`), using the gradient of the error with respect to the output (`δEδy`). If the model/layer has parameters, it also computes the gradient with respect to those parameters.
    * `backward` makes it possible to write a decoupled implementation of the backpropagation algorithm.
    * Using a model's `backward`, that model can be optimized with gradient descent.


The library provides several classes for different layers/models:

* The `Linear` and `Bias` classes, which create layers with the functions $wx$ and $x+b$ respectively. In these cases, $w$ and $b$ are the parameters to be optimized. By combining these layers you can build a traditional dense layer that computes $wx+b$.
* The `TanH`, `ReLU` and `Softmax` classes, which create layers with the activation functions of those names.
* The `Sequential` class, for creating sequential networks, where the output of each layer is the input of the next one, and there is a single initial layer and a single final layer.

Every one of these classes is a subclass of `Model`, and therefore supports the 2 fundamental operations, `forward` and `backward`. From now on we will use the word _layer_ as a synonym for model, that is, for a subclass of `Model`. This terminology, while somewhat imprecise, is standard in the field of neural networks.


In [ ]:
layers = [nn.Linear(din,10),
          nn.Bias(10),
          nn.ReLU(),
          nn.Linear(10,20),
          nn.Bias(20),
          nn.TanH(),
          nn.Linear(20,n_classes),
          nn.Bias(n_classes),
          nn.Softmax()
          ]
model = nn.Sequential(layers)
print("Model summary:")
print(model.summary())

# Error layers/models

Models need their error to be measured. For that, `edunn` also provides some error layers/models, whose `forward` takes two inputs: the one computed by the network and the expected one. There are two kinds of layers:

* Those that compute the network's error _for each example separately_, such as `CrossEntropyWithLabels` or `SquaredError`
* Those that combine the errors of each example to produce a scalar error.
    * The `MeanError` layer computes the mean error of another error layer, such as the `CrossEntropyWithLabels` and `SquaredError` layers we just mentioned.
    
    

In [ ]:
mean_cross_entropy_error = nn.MeanError(nn.CrossEntropyWithLabels())
mean_squared_error = nn.MeanError(nn.SquaredError())

# Optimizers 

To train a model we can use an `Optimizer` object, whose `optimize` method takes arrays `x` and `y` and an error function, and trains a model to minimize that error on this dataset. 
For this training, an optimization algorithm must be specified. In this case we use plain gradient descent with the `GradientDescent` class, a learning rate of `0.1`, `100` epochs and a batch size of 8.


In [ ]:

# Optimization algorithm
optimizer = nn.GradientDescent(lr=0.001,epochs=100,batch_size=8)

# Optimization
history = optimizer.optimize(model,x,y,mean_cross_entropy_error)



Finally, we can use and evaluate the model:
* The `forward` method gives you a model's output. 
    * For the Sequential class, which is made up of several layers, `forward` returns the output of the last layer, without the error
    * For a classification problem, we must compute the argmax, since the output is the class probabilities for each example.

In addition, `edunn` has some extra functionality to make working with networks easier:

* The `metrics` module has some functions for evaluating the model's performance metrics.
* The `plot` module has some functions for monitoring the model's training (`plot_history`) and, when the problem is low-dimensional (1 or 2), also for visualizing the decision boundaries or the fitted function (`plot_model_dataset_2d_classification`)



In [ ]:
nn.plot.plot_history(history)

# Performance report
y_pred = model.forward(x)
y_pred_labels = y_pred.argmax(axis=1)
print(f"Final model accuracy: {nn.metrics.accuracy(y,y_pred_labels)*100:0.2f}%")


if din ==2:
    # Model visualization, only if it has 2 dimensions
    nn.plot.plot_model_dataset_2d_classification(x,y,model,title=dataset_name)



As you may have noticed, even though we were able to define the network and call the `optimize` method to ask the model to train itself with gradient descent, the accuracy we obtained is very poor: the network is not learning! 

This is _not_ a bug: none of the corresponding methods of the models (Bias, Linear, etc) are implemented, and neither is the `GradientDescent` optimizer. 

**Your** task will be to implement the different layers/models of the `edunn` library, as well as some initializers and optimization algorithms, so that this code works.


# Reference implementation 

The [edunn repository](https://github.com/facundoq/edunn) contains a reference implementation that focuses on being easy to understand rather than on computational efficiency.

Starting from the code of that reference implementation, and using a program that processes it, a version of edunn was generated in which crucial parts of the implementation of each layer and of other classes were removed.

To reimplement the library, you will have to look for the lines of code between the comments 

```""" YOUR IMPLEMENTATION START """```

and

```""" YOUR IMPLEMENTATION END """```

 and fill in the corresponding code.

In every case, it is important to focus on finding an implementation that is easy to understand and correct, and to leave efficiency aside for a later implementation.

Although this implementation guide was originally written in Spanish, the library itself is implemented in English so that it is easier to relate its concepts to those of other libraries.

The following notebooks will guide you through the implementation of each `Model', covering both the forward and the backward methods, as well as important methods of other classes.

Whenever you are in doubt, you can always look up one possible solution in the [reference implementation](https://github.com/facundoq/edunn/tree/main/edunn).

# IMPLEMENTATION plan

You will begin by implementing two very simple *layers*: `AddConstant` and `MultiplyConstant`. The word *layer* is roughly equivalent to *model*, but it emphasizes that the model is meant to be combined with another *layer* to form a larger *model*.


In particular, these two layers perform very simple functions and have no parameters:

* `AddConstant` adds a constant to a matrix
* `MultiplyConstant` multiplies a matrix by a constant.

Implementing the corresponding `forward` and `backward` methods will therefore be straightforward, and it will let you start getting familiar with "edunn" and with the methodology.

After that, we will start implementing more complex layers, such as `Bias` and `Linear`, to build `LinearRegression` models, together with their most common error function, `SquaredError`, and a `MeanError` to average the per-sample errors over a batch of samples.
At that point we will also implement a `GradientDescent` optimizer to put our models to the test. After that, we will dive into more complex layers, such as "Softmax", to perform classification. Finally, we will implement the "Sequential" model to compose several layers into a complete modular neural network.

We hope you have a lot of fun programming your first modular neural network! 🕸️🚀